# Stage 6 - LLM-as-Judge

## Data
Chroma DB

## install
pip install pandas

In [2]:
'''
1.1 Vectors DB Client
'''
import chromadb

client = chromadb.PersistentClient(path="chroma_kb") 
col = client.get_collection(name="knowledge_base")         

In [3]:
'''
1.2 Retrieve 
'''
from sentence_transformers import SentenceTransformer

THRESHOLD = 0.8
K = 5

embedder = SentenceTransformer("intfloat/multilingual-e5-base", device="mps")

def dense_search(query, col=col, embedder=embedder, k=K, threshold=THRESHOLD):
  """Retrieval-пайплайн: embed query -> search -> filter -> return."""
  q_vec = embedder.encode("query: " + query, normalize_embeddings=True)
  res = col.query(query_embeddings=[q_vec.tolist()], n_results=k)
  out = []
  for doc, m, dist in zip(res["documents"][0],
                          res["metadatas"][0],
                          res["distances"][0]):
    sim = 1.0 - dist                        # Косинусное сходство
    if sim >= threshold:                    # Порог по сходству
      out.append({
        'id': m['id'],
        'doc_id': m['doc_id'], 
        'similarity': round(sim, 4), 
        'text': doc, 
      }
    )
  return out

/Users/german/work/yai_02/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W1116 08:34:08.826000 31156 torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 7449.25it/s]


In [21]:
'''
1.3 load queries
'''
import json

with open("1_queries.json", encoding="utf-8") as f:
  queries = json.load(f)

In [5]:
'''
2. LLM Generate
'''
from openai import OpenAI

client = OpenAI(base_url="http://localhost:8080/v1", api_key="not-needed")
LLM = "qwen3.5"                                       # Имя сервер игнорирует
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}


def generate(messages, max_tokens=160):
    """Один вызов локальной LLM. temperature=0 — для воспроизводимости,
    режим рассуждения Qwen3.5 отключён, поэтому сразу приходит готовый текст."""
    resp = client.chat.completions.create(
        model=LLM, messages=messages, temperature=0.0,
        max_tokens=max_tokens, extra_body=NO_THINK)
    return resp.choices[0].message.content.strip()

In [6]:
SYSTEM = ("Ты — ассистент поддержки. Отвечай ТОЛЬКО по приведённым фрагментам "
          "базы знаний. Если ответа в них нет, ответь: «В базе знаний ответа "
          "нет». Отвечай кратко, одним-двумя предложениями.")


def build_messages(query, chunk_dicts):
    """Собирает промпт: system-инструкция + нумерованный контекст + вопрос."""
    # print('>>> chunk_dicts: ', chunk_dicts)
    context = "\n".join(f"[{n}] {i['text']}" for n, i in enumerate(chunk_dicts, 1))
    user = f"Фрагменты базы знаний:\n{context}\n\nВопрос: {query}"
    return [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": user}]

def rag_answer(query, search_fn, k=5):
    """Полный RAG: retrieval -> промпт -> генерация."""
    ret = search_fn(query, k=5)
    msg = build_messages(query, ret)
    answer = generate(msg)
    return answer, [i['doc_id'] for i in ret], [i['text'] for i in ret]

In [9]:
JUDGE_SYSTEM = (
    "Ты — строгий эксперт по оценке ответов справочных систем. Оценивай "
    "беспристрастно, длина ответа не является достоинством. "
    "Отвечай только JSON-объектом."
)

FAITH_PROMPT = """Оцени ВЕРНОСТЬ КОНТЕКСТУ (faithfulness):
следует ли каждое утверждение ответа из фрагментов.
Полезность ответа и его соответствие вопросу не оценивай —
только происхождение фактов.

Шкала:
5 — все утверждения следуют из фрагментов;
4 — все утверждения следуют из фрагментов,
добавлено безобидное обобщение;
3 — главное утверждение из фрагментов,
но добавлена деталь, которой в них нет;
2 — часть утверждений выдумана или противоречит фрагментам;
1 — ключевые факты ответа отсутствуют во фрагментах или искажены.
Ответ «В базе знаний ответа нет» не утверждает ничего и получает 5.

Фрагменты:
{context}

Ответ: {answer}

Верни JSON с двумя полями:
{{"reason": "<одно предложение>", "score": <целое число 1-5>}}"""

In [22]:
'''
load judge_cases
'''
import json

with open("6_judge_cases.json", encoding="utf-8") as f:
  judge_cases = json.load(f)

with open("2_chunks.json", encoding="utf-8") as f:
  chunks = json.load(f)

texts = [c["text"] for c in chunks]

In [ ]:
# Используем client, LLM и NO_THINK из общего кода

JUDGE_SYSTEM = (
    "Ты — строгий эксперт по оценке справочных ответов. Оценивай "
    "беспристрастно, длина ответа не является достоинством. "
    "Отвечай только JSON-объектом."
)

FAITH_PROMPT = """Оцени ВЕРНОСТЬ КОНТЕКСТУ (faithfulness):
следует ли каждое утверждение ответа из фрагментов.
Соответствие вопросу не оценивай — только происхождение фактов.

Шкала:
5 — все утверждения следуют из фрагментов;
4 — добавлено только безобидное обобщение;
3 — добавлена неподтверждённая деталь;
2 — часть утверждений выдумана или противоречит фрагментам;
1 — ключевые факты отсутствуют во фрагментах или искажены.
Ответ «В базе знаний ответа нет» ничего не утверждает и получает 5.

Фрагменты:
{context}

Ответ: {answer}

Верни JSON с полями reason и score."""

REL_PROMPT = """Оцени РЕЛЕВАНТНОСТЬ (answer relevance):
отвечает ли ответ на заданный вопрос по существу.
Правдивость не проверяй — только соответствие вопросу.

Шкала:
5 — прямой и полный ответ;
4 — ответ по существу с небольшими отступлениями;
3 — закрыта только часть вопроса;
2 — текст на нужную тему, но ответа нет;
1 — другая тема или отказ отвечать.

Вопрос: {question}

Ответ: {answer}

Верни JSON с полями reason и score."""

JUDGE_RESPONSE_FORMAT = {
    "type": "json_object",
    "schema": {
        "type": "object",
        "properties": {
            "reason": {"type": "string", "maxLength": 240},
            "score": {
                "type": "integer",
                "minimum": 1,
                "maximum": 5,
            },
        },
        "required": ["reason", "score"],
        "additionalProperties": False,
    },
}


def judge(prompt):
    """Возвращает оценку 1–5 и обоснование LLM-судьи."""
    response = client.chat.completions.create(
        model=LLM,
        messages=[
            {"role": "system", "content": JUDGE_SYSTEM},
            {"role": "user", "content": prompt},
        ],
        temperature=0.0,
        max_tokens=200,
        response_format=JUDGE_RESPONSE_FORMAT,
        extra_body=NO_THINK,
    )
    value = json.loads(response.choices[0].message.content)
    score = value.get("score")
    reason = value.get("reason")
    if type(score) is not int or not 1 <= score <= 5:
        raise ValueError(f"Некорректная оценка судьи: {score!r}")
    if not isinstance(reason, str) or not reason.strip():
        raise ValueError("Судья не вернул обоснование")
    return score, reason.strip()

# кортеж «(оценка, обоснование)»
def judge_faithfulness(context, answer):
    """Оценивает верность ответа контексту."""
    prompt = FAITH_PROMPT.format(context=context, answer=answer)
    return judge(prompt)

# кортеж «(оценка, обоснование)»
def judge_relevance(question, answer):
    """Оценивает соответствие ответа вопросу."""
    prompt = REL_PROMPT.format(question=question, answer=answer)
    return judge(prompt)

def build_context(chunk_ids):
    """Склеивает чанки в нумерованный контекст."""
    return "\n".join(f"[{n}] {texts[i]}" for n, i in enumerate(chunk_ids, 1))

In [29]:
for case in judge_cases:
  print('>>> case: ', case['query'])
  context = build_context(case["context_ids"])
  print(judge_faithfulness(context, case["answer"]))
  print(judge_relevance(case["query"], case["answer"]))
  print('\n')

>>> case:  что означает ошибка P-102
(5, 'Ответ содержит все ключевые факты из фрагмента [1]: причина ошибки (превышение лимита одновременных просмотров), необходимость проверки количества экранов, обновление приложения и переход на тариф с большим лимитом. Дополнительные детали (у')
(5, 'Ответ полностью соответствует вопросу, объясняя причину возникновения ошибки P-102 (превышение лимита просмотров) и предоставляя конкретные шаги по её устранению. Информация является прямой и исчерпывающей для данного запроса.')


>>> case:  какая версия webOS или Tizen нужна для приложения на телевизоре
(5, 'Утверждения в ответе полностью соответствуют информации, содержащейся во фрагментах [1] и [2]. Фрагменты явно указывают, что для Samsung требуется Tizen с моделями с 2018 года, а для LG — webOS 4.0 и новее. Никаких фактов не добавлено, иска')
(5, 'Ответ содержит прямую и конкретную информацию, касающуюся требуемых версий операционных систем (Tizen и webOS) для приложений на телевизорах разных бре

In [31]:
import numpy as np

# Четыре параллельных списка: по одному значению на каждый из 10 кейсов.
# judge_* — оценки нашего судьи, human_* — оценки асессора из corpus.json
judge_faith, judge_rel = [], []
human_faith, human_rel = [], []
# judged_rows — оценки обоих судей вместе с обоснованиями,
# чтобы потом напечатать проблемные ответы
judged_rows = []

for case in judge_cases:
    context = build_context(case["context_ids"])
    faith, faith_reason = judge_faithfulness(context, case["answer"])
    rel, rel_reason = judge_relevance(case["query"], case["answer"])
    judge_faith.append(faith)
    judge_rel.append(rel)
    human_faith.append(case["human_faithfulness"])
    human_rel.append(case["human_relevance"])
    judged_rows.append(
        {
            "id": case["id"],
            "faith": faith,
            "rel": rel,
            "faith_reason": faith_reason,
            "rel_reason": rel_reason,
        }
    )


def decision_agreement(scores, human_scores, threshold):
    """Считает совпадения решений по заданному порогу."""
    return sum(
        (score < threshold) == (human < threshold)
        for score, human in zip(scores, human_scores)
    )


print("Средний faithfulness:", round(float(np.mean(judge_faith)), 1))
print("Средний answer relevance:", round(float(np.mean(judge_rel)), 1))
print(
    "Точное совпадение faith:",
    sum(a == b for a, b in zip(judge_faith, human_faith)),
)
print(
    "Точное совпадение relevance:",
    sum(a == b for a, b in zip(judge_rel, human_rel)),
)
for threshold in (3, 4):
    print(
        "Порог ниже", threshold,
        "| faith", decision_agreement(judge_faith, human_faith, threshold),
        "| relevance", decision_agreement(judge_rel, human_rel, threshold),
    )

print("Проблемные ответы по рабочему порогу:")
for row in judged_rows:
    if row["faith"] < 4 or row["rel"] < 3:
        print(row)

Средний faithfulness: 5.0
Средний answer relevance: 3.7
Точное совпадение faith: 2
Точное совпадение relevance: 3
Порог ниже 3 | faith 2 | relevance 3
Порог ниже 4 | faith 2 | relevance 3
Проблемные ответы по рабочему порогу:
{'id': 3, 'faith': 5, 'rel': 1, 'faith_reason': "Ответ 'В базе знаний ответа нет' является корректным, так как ни одно из утверждений в ответе не содержит фактов, которые можно было бы проверить на соответствие предоставленным фрагментам. Фрагменты содержат подробную информацию о тарифах,", 'rel_reason': 'Ответ прямо указывает на отсутствие информации в базе знаний, что делает невозможным提供任何 релевантную информацию по существу вопроса.'}


# Threshold = 3

In [35]:
import json

with open("1_corpus.json", encoding="utf-8") as f:
    docs = json.load(f)

print("Разделов:", len(docs), "| Запросов:", len(queries))
print(queries[0])

Разделов: 154 | Запросов: 66
{'query': 'что означает ошибка P-102', 'kind': 'exact', 'relevant_docs': [85], 'gold': 'превышено число одновременных просмотров по тарифу; нужно остановить просмотр на другом устройстве или перейти на тариф с большим лимитом'}


In [38]:
import numpy as np
from openai import OpenAI
from sentence_transformers import SentenceTransformer


def naive_split(text, size=120):
    """Разрезает text на фрагменты по size символов без перекрытия.

    Возвращает список строк; границы предложений не учитываются.
    """
    return [text[i:i + size] for i in range(0, len(text), size)]


chunks, chunk_doc = [], []                # Тексты чанков и раздел-источник
for d in docs:
    for piece in naive_split(d["raw"]):
        chunks.append(piece)
        chunk_doc.append(d["doc_id"])     # Чанк помнит, из какого раздела вырезан

print("Чанков:", len(chunks))             # Чанков: 108

# Эмбеддер темы: multilingual-e5-base с префиксами query: и passage:
embedder = SentenceTransformer("intfloat/multilingual-e5-base", device="mps")
doc_vecs = embedder.encode(["passage: " + c for c in chunks],
                           normalize_embeddings=True, show_progress_bar=False)


def search(query, k=5):
    """Ищет для query ближайшие k чанков.

    Возвращает тексты чанков и идентификаторы разделов-источников.
    """
    q_vec = embedder.encode("query: " + query, normalize_embeddings=True)
    top = np.argsort(q_vec @ doc_vecs.T)[::-1][:k]
    return [chunks[i] for i in top], [chunk_doc[i] for i in top]


# Генерация: тот же локальный сервер и тот же заземляющий промпт
client = OpenAI(base_url="http://localhost:8080/v1", api_key="not-needed")
NO_THINK = {"chat_template_kwargs": {"enable_thinking": False}}

SYSTEM = ("Ты — ассистент поддержки. Отвечай ТОЛЬКО по приведённым фрагментам "
          "базы знаний. Если ответа в них нет, ответь: «В базе знаний ответа "
          "нет». Отвечай кратко, одним-двумя предложениями.")


def generate(messages, max_tokens=200):
    """Отправляет messages локальной модели и ограничивает ответ max_tokens.

    Возвращает очищенный текст ответа; temperature=0 — для воспроизводимости.
    """
    resp = client.chat.completions.create(
        model="qwen3.5", messages=messages, temperature=0.0,
        max_tokens=max_tokens, extra_body=NO_THINK)
    return resp.choices[0].message.content.strip()


def rag_answer(query, k=5):
    """Запускает RAG для query по k найденным чанкам.

    Возвращает ответ, тексты чанков и идентификаторы разделов-источников.
    """
    ctx, got_docs = search(query, k)
    numbered = "\n".join(f"[{i}] {c}" for i, c in enumerate(ctx, 1))
    answer = generate([
        {"role": "system", "content": SYSTEM},
        {"role": "user", "content":
         f"Фрагменты базы знаний:\n{numbered}\n\nВопрос: {query}"}])
    return answer, ctx, got_docs

Чанков: 1049


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 25951.95it/s]


In [39]:
import re

JUDGE_FAITH = (
    "Ты — строгий эксперт по оценке ответов. Оцени, насколько ОТВЕТ опирается "
    "только на ФРАГМЕНТЫ. 5 — каждое утверждение ответа есть во фрагментах; "
    "1 — ответ содержит факты, которых во фрагментах нет. Ответ «в базе знаний "
    "ответа нет» считай полностью опирающимся на фрагменты (5). "
    "Верни только одну цифру от 1 до 5.")

JUDGE_RELEV = (
    "Ты — строгий эксперт по оценке ответов. Оцени, отвечает ли ОТВЕТ по "
    "существу на ВОПРОС. 5 — прямой и полный ответ на заданный вопрос; "
    "1 — ответ не по теме или отказ отвечать. Верни только одну цифру от 1 до 5.")

JUDGE_CORRECT = (
    "Ты — строгий эксперт. Сравни ОТВЕТ с ЭТАЛОННЫМ ФАКТОМ. Верни 1, если ответ "
    "передаёт эталонный факт (формулировка может отличаться), и 0, если ответ "
    "противоречит эталону, неполон или отказывается отвечать. "
    "Верни только одну цифру: 1 или 0.")


def judge(question, context, answer, gold):
    """Оценивает answer для question по context и эталонному gold.

    Возвращает faithfulness, answer relevance и бинарную correctness.
    """
    ctx = "\n".join(f"[{i}] {c}" for i, c in enumerate(context, 1))
    faith = generate([{"role": "system", "content": JUDGE_FAITH},
                      {"role": "user", "content":
                       f"ФРАГМЕНТЫ:\n{ctx}\n\nОТВЕТ: {answer}"}], max_tokens=8)
    relev = generate([{"role": "system", "content": JUDGE_RELEV},
                      {"role": "user", "content":
                       f"ВОПРОС: {question}\n\nОТВЕТ: {answer}"}], max_tokens=8)
    correct = generate([{"role": "system", "content": JUDGE_CORRECT},
                        {"role": "user", "content":
                         f"ВОПРОС: {question}\nЭТАЛОННЫЙ ФАКТ: {gold}\n\n"
                         f"ОТВЕТ: {answer}"}], max_tokens=8)
    return (int(re.search(r"[1-5]", faith).group()),
            int(re.search(r"[1-5]", relev).group()),
            int(re.search(r"[01]", correct).group()))

In [52]:
import pandas as pd

# Широкий вывод: иначе pandas свернёт часть столбцов в многоточие
pd.set_option("display.width", 120)
pd.set_option("display.max_columns", None)


def precision_at_k(got_docs, relevant):
    """Доля слотов выдачи, занятых уникальными релевантными разделами."""
    # Пересечение множеств: два чанка одного раздела дают один релевантный
    # раздел, а знаменатель — размер выдачи, то есть K
    return len(set(got_docs) & set(relevant)) / len(got_docs) if got_docs else 0.0


def recall_at_k(got_docs, relevant):
    """Доля релевантных разделов, которые поиск донёс до промпта."""
    # Знаменатель другой — все релевантные разделы из разметки
    return sum(d in got_docs for d in relevant) / len(relevant) if relevant else 0.0


def mrr(got_docs, relevant):
    """Обратный ранг первого релевантного раздела."""
    for rank, doc in enumerate(got_docs, start=1):
        if doc in relevant:
            return 1 / rank      # Первое место — 1, второе — 0,5, третье — 1/3
    return 0.0                   # Релевантных разделов в выдаче нет вовсе


# Прогон по всем 30 запросам: на каждый — ответ системы и три оценки судьи
rows = []
brief = [i for i in queries if i['kind'] == 'exact'][:6] \
  + [i for i in queries if i['kind'] == 'semantic'][:6] \
  + [i for i in queries if i['kind'] == 'multi'][:6] \
  + [i for i in queries if i['kind'] == 'gap'][:6] 

count = 0
for q in brief:
  count += 1
  answer, ctx, got_docs = rag_answer(q["query"], k=5)
  faith, relev, correct = judge(q["query"], ctx, answer, q["gold"])
  rows.append({
      "kind": q["kind"],                # Тип запроса для разбивки по срезам
      "query": q["query"],
      # Метрики поиска считаем по разделам-источникам найденных чанков
      "precision@5": precision_at_k(got_docs, q["relevant_docs"]),
      "recall@5": recall_at_k(got_docs, q["relevant_docs"]),
      "MRR": mrr(got_docs, q["relevant_docs"]),
      # Метрики генерации: их проставил судья
      "faithfulness": faith,
      "answer relevance": relev,
      "correctness": correct,
      "answer": answer,                 # Ответ и контекст сохраняем, чтобы
      "ctx": ctx,                       # разобрать провалы глазами в следующем задании
  })
  # print('>>> # ', count, q)

report = pd.DataFrame(rows)

# Только числовые столбцы: по ним и считаем средние
METRICS = ["precision@5", "recall@5", "MRR", "faithfulness",
           "answer relevance", "correctness"]

print("Сводный профиль:")
print(report[METRICS].mean().round(3))          # Средние по всему набору
print("\nПо типам запросов:")
print(report.groupby("kind")[METRICS].mean().round(3))   # Срезы по полю kind

Сводный профиль:
precision@5         0.133
recall@5            0.479
MRR                 0.382
faithfulness        4.958
answer relevance    5.000
correctness         0.750
dtype: float64

По типам запросов:
          precision@5  recall@5    MRR  faithfulness  answer relevance  correctness
kind                                                                               
exact           0.200     1.000  0.833         5.000               5.0          1.0
gap             0.000     0.000  0.000         5.000               5.0          1.0
multi           0.300     0.750  0.639         4.833               5.0          0.5
semantic        0.033     0.167  0.056         5.000               5.0          0.5
